# Let's group IRIS data! 
In this notebook we will do again some analysis on IRIS but using grouping functions with Pandas

## Exercise 1 - Load the dataset
Download the dataset and read it directly with pandas into a dataframe with the pd.read_csv function directly from:
"https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"

Remember: the file has no header, so you need to specify it when you read it.
Since it has no header let's assign the following column names to it:
- sepal_length
- sepal_width
- petal_length
- petal_width
- label

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 400)

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "label"]
df = pd.read_csv(url, header=None, names=columns)

print(df.head())  # prime 5 righe
print(df.shape)  # dovrebbe essere (150, 5)

   sepal_length  sepal_width  petal_length  petal_width        label
0           5.1          3.5           1.4          0.2  Iris-setosa
1           4.9          3.0           1.4          0.2  Iris-setosa
2           4.7          3.2           1.3          0.2  Iris-setosa
3           4.6          3.1           1.5          0.2  Iris-setosa
4           5.0          3.6           1.4          0.2  Iris-setosa
(150, 5)


## Exercise 2 - compute the mean and std 
Compute the mean and std of each feature for each class by grouping the classes

In [ ]:
mean = df.groupby("label").mean()  # Dataframe
std = df.groupby("label").std()  # Dataframe

print(f"Mean =\n{mean}")
print("\n")
print(f"Std =\n{std}")

Mean =
                 sepal_length  sepal_width  petal_length  petal_width
label                                                                
Iris-setosa             5.006        3.418         1.464        0.244
Iris-versicolor         5.936        2.770         4.260        1.326
Iris-virginica          6.588        2.974         5.552        2.026


Std =
                 sepal_length  sepal_width  petal_length  petal_width
label                                                                
Iris-setosa          0.352490     0.381024      0.173511     0.107210
Iris-versicolor      0.516171     0.313798      0.469911     0.197753
Iris-virginica       0.635880     0.322497      0.551895     0.274650


## $(\star)$ Exercise 3 - predict the class of a sample by computing the distance from the mean of each class
- Create a function to compute the  distance of a sample from the mean of each class 
- Then create a new column in the dataset called "predicted_label" and assign the predicted class to each sample
- Then compute the accuracy of the prediction

**Note:** you must not iterate over the dataframe


In [72]:
# Nearest centroid classifier:
# each sample is assigned to the class whose mean (centroid) is closest to it.

# Keep only the four numeric features: the label is the answer we want to predict,
# so it must not be used to compute distances (and strings can't be subtracted anyway).
features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
df_nolabel = df[features]
print("Dataframe without label =")
print(df_nolabel.head())


def distance_from_mean(samples, class_mean):
    """Return the Euclidean distance of every sample from one class mean."""
    # Subtract the class mean from every row (pandas aligns on column names).
    diff = samples - class_mean
    # Square each difference and add them up across the columns (axis=1),
    # giving one value per sample.
    squared_sum = (diff**2).sum(axis=1)
    # Take the square root to get the Euclidean distance.
    distances = squared_sum**0.5
    return distances


# Compute the distance of every sample from each of the three class means.
# The loop runs only 3 times (once per class), not once per sample.
distances = {}
for class_name in mean.index:
    distances[class_name] = distance_from_mean(df_nolabel, mean.loc[class_name])

# Turn the dictionary into a table: one row per sample, one column per class.
distances_df = pd.DataFrame(distances)
print(f"\nDistances Dataframe =\n{distances_df.head()}")
print(f"\nDistances Shape =\n{distances_df.shape}")

# For each sample (row), pick the name of the column with the smallest distance:
# that is the predicted class.
df["predicted_label"] = distances_df.idxmin(axis=1)
print(f"\nDataframe with Predictions =\n{df.head()}")

# Check accuracy:
# compare true and predicted labels row by row (True if they match).
correct = df["label"] == df["predicted_label"]
# The mean of True/False values is the fraction of correct predictions.
accuracy = correct.mean()
print(f"\nAccuracy = {accuracy:.4f}")

Dataframe without label =
   sepal_length  sepal_width  petal_length  petal_width
0           5.1          3.5           1.4          0.2
1           4.9          3.0           1.4          0.2
2           4.7          3.2           1.3          0.2
3           4.6          3.1           1.5          0.2
4           5.0          3.6           1.4          0.2

Distances Dataframe =
   Iris-setosa  Iris-versicolor  Iris-virginica
0     0.146942         3.267916        4.802520
1     0.438169         3.251718        4.839773
2     0.412301         3.426671        5.002939
3     0.518837         3.283180        4.870421
4     0.197970         3.318504        4.846339

Distances Shape =
(150, 3)

Dataframe with Predictions =
   sepal_length  sepal_width  petal_length  petal_width        label predicted_label
0           5.1          3.5           1.4          0.2  Iris-setosa     Iris-setosa
1           4.9          3.0           1.4          0.2  Iris-setosa     Iris-setosa
2           4.